# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hafsaShaban/flyrank_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists('/content/flyrank_internship'):
    !git clone https://github.com/hafsaShaban/flyrank_internship.git
%cd /content/flyrank_internship

import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print("Loaded:", len(df), "rows")

Cloning into 'flyrank_internship'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 131 (delta 43), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 1.83 MiB | 16.86 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/flyrank_internship
Loaded: 30000 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
print("Rows:", len(df))
print("Unique content_id:", df['content_id'].nunique())
print("Unique client_id:", df['client_id'].nunique())

Rows: 30000
Unique content_id: 30000
Unique client_id: 32


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
feature_cols = ['word_count','char_count','ctr','avg_position','engagement_rate','scroll_rate',
    'ai_traffic_pct','content_age_days','days_since_last_update','freshness_tier','word_count_tier',
    'char_count_tier','impression_tier','position_tier','search_volume','competition','competition_level',
    'cpc','content_type','main_intent','impressions_90d','clicks_90d','pageviews_90d','sessions_90d',
    'users_90d','engaged_sessions_90d','ai_sessions_90d','scroll_events_90d','days_with_impressions',
    'days_with_sessions','impressions_prev_30d','clicks_prev_30d','sessions_prev_30d']
label_cols = ['trend_direction','trend_pct','is_declining_label']
context_cols = ['content_id','client_id']
excluded_cols = ['impressions_last_30d','clicks_last_30d','sessions_last_30d','provider_used','model_used']

all_cols = set(df.columns) | {'is_declining_label'}
classified = set(feature_cols) | set(label_cols) | set(context_cols) | set(excluded_cols)
print("Unclassified columns:", all_cols - classified)
print("Overlap between buckets (should be empty):",
      set(feature_cols) & set(label_cols) & set(context_cols) & set(excluded_cols))

Unclassified columns: {'age_tier', 'age_tier_order'}
Overlap between buckets (should be empty): set()


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# Grain check
dupes = df.groupby('content_id').size()
print("content_id rows appearing more than once:", (dupes > 1).sum())

# Counts per client
print(df.groupby('client_id').size().describe())

# Missingness overall
print(df.isnull().mean().sort_values(ascending=False).head(10))

# Missingness by content_type (checking for patterned, not random, gaps)
print(df.groupby('content_type')['word_count'].apply(lambda x: x.isnull().mean()))

content_id rows appearing more than once: 0
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64
provider_used        0.714600
word_count           0.256633
char_count           0.256633
char_count_tier      0.256633
word_count_tier      0.256633
model_used           0.191100
trend_pct            0.112933
competition_level    0.087000
cpc                  0.082267
search_volume        0.082267
dtype: float64
content_type
comparison article    0.000000
feedly article        0.000000
keyword article       0.282979
Name: word_count, dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [5]:
print("avg_position == 0 (no data) rows:", (df['avg_position'] == 0).sum())
print("Max scroll_rate:", df['scroll_rate'].max())
print("Max ai_traffic_pct:", df['ai_traffic_pct'].max())
print("Sample ctr values:", df['ctr'].head())

avg_position == 0 (no data) rows: 1205
Max scroll_rate: 300.0
Max ai_traffic_pct: 300.0
Sample ctr values: 0    0.76
1    0.05
2    0.09
3    0.49
4    0.13
Name: ctr, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.